[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samcho93/studyAgent/blob/main/notebooks/11_project_assistant.ipynb)

> 📘 **학생용 실습 노트북** · 강의 페이지: [11차시 강의 보기](https://samcho93.github.io/studyAgent/student.html#ag11)
>
> 1. 본인 **Google 계정**으로 로그인한 뒤, 메뉴 **파일 → Drive에 사본 저장**을 눌러 내 드라이브에 복사해서 사용하세요.
> 2. 셀을 위에서부터 차례로 실행하세요 (**Shift + Enter**).
> 3. `✏️ 여기에 코드를 작성하세요` 또는 `___` 표시가 있는 곳이 직접 풀어볼 실습 문제입니다.

# 11. 프로젝트 ①: 날씨 · 검색 비서 에이전트 — LangChain 으로 완성하기

브라우저에서 `agentlab` 으로 만든 비서를 **실제 LangChain + LangGraph** 로 다시 만듭니다.

| 단계 | 내용 | 브라우저 예제 |
|---|---|---|
| 1 | 도구 단독 테스트 (DuckDuckGo · Open-Meteo / OpenWeatherMap · 옷차림) | 11-1 ~ 11-3 |
| 2 | `create_react_agent` 로 조립 · 페르소나 · 첫 질문 | 11-4 ~ 11-6 |
| 3 | `MemorySaver` 로 멀티턴 대화 | 11-7 |
| 4 | 오류 처리 도구 · 두 도구 연쇄 | 11-8 ~ 11-9 |
| 5 | 대화 로그 마크다운 저장 · CLI 루프 | 11-10 ~ 11-11 |
| ✏️ | 실습 문제 4개 | |

**API 키**: 왼쪽 🔑 Secrets 에 `GEMINI_API_KEY`(필수, [aistudio.google.com](https://aistudio.google.com/apikey) 무료) · `OPENWEATHER_API_KEY`(선택) 를 넣고 **노트북 액세스**를 켜세요.

## 0. 설치와 키 설정

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-google-genai langgraph duckduckgo-search requests

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")          # LangChain Gemini 는 GOOGLE_API_KEY 를 읽는다
try:
    os.environ["OPENWEATHER_API_KEY"] = userdata.get("OPENWEATHER_API_KEY")
    print("OpenWeatherMap 키 있음")
except Exception:
    print("OpenWeatherMap 키 없음 → Open-Meteo(키 불필요)로 대체합니다")

from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print(llm.invoke("한 문장으로 자기소개 해줘").content)

---
## 1단계. 도구 단독 테스트
📘 브라우저에서와 같은 순서입니다: **도구를 먼저**, 에이전트는 나중에. `@tool` 의 docstring 이 LLM 에게 보이는 설명입니다.

In [ ]:
import requests
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> dict:
    """도시의 현재 날씨(기온 · 상태 · 습도)를 알려준다. city: 도시 이름 (영문 권장, 예: Busan, Seoul)"""
    key = os.environ.get("OPENWEATHER_API_KEY")
    if key:                                                            # OpenWeatherMap (무료 키)
        r = requests.get("https://api.openweathermap.org/data/2.5/weather",
                         params={"q": city, "appid": key, "units": "metric", "lang": "kr"}, timeout=10).json()
        if r.get("cod") != 200:
            return {"error": f"{city}: {r.get('message', '조회 실패')}"}
        return {"city": city, "temperature": r["main"]["temp"], "condition": r["weather"][0]["description"],
                "humidity": r["main"]["humidity"], "source": "openweathermap"}
    g = requests.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}, timeout=10).json()
    if not g.get("results"):
        return {"error": f"'{city}' 도시를 찾지 못했습니다"}
    loc = g["results"][0]
    c = requests.get("https://api.open-meteo.com/v1/forecast",
                     params={"latitude": loc["latitude"], "longitude": loc["longitude"],
                             "current": "temperature_2m,relative_humidity_2m,weather_code"}, timeout=10).json()["current"]
    return {"city": loc["name"], "temperature": c["temperature_2m"], "humidity": c["relative_humidity_2m"],
            "weather_code": c["weather_code"], "source": "open-meteo"}

print(get_weather.invoke({"city": "Busan"}))
print(get_weather.invoke({"city": "Atlantis"}))          # 없는 도시 → error 키
print(get_weather.name, "|", get_weather.description)

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()                            # 키 불필요 웹 검색
print(search.invoke("전기차 보조금 2026")[:300])

In [ ]:
@tool
def recommend_outfit(temperature: float) -> dict:
    """기온(섭씨)에 맞는 옷차림을 추천한다. temperature: 섭씨 기온 (숫자)"""
    t = float(temperature)
    tip = "반팔과 반바지" if t >= 28 else "얇은 긴팔" if t >= 20 else "가벼운 재킷" if t >= 12 else "코트와 니트" if t >= 5 else "두꺼운 패딩"
    return {"temperature": t, "outfit": tip}

@tool
def calculator(expression: str) -> dict:
    """수식을 계산한다. expression: 예 '21 * 9/5 + 32'"""
    import math, re
    if not re.fullmatch(r"[\d\s\.\+\-\*/\(\)]+", expression):
        return {"error": "허용되지 않는 문자"}
    return {"expression": expression, "result": eval(expression, {"__builtins__": {}}, {})}

print(recommend_outfit.invoke({"temperature": 18.4}))
print(calculator.invoke({"expression": "21 * 9/5 + 32"}))

---
## 2단계. 에이전트 조립 — `create_react_agent`
📘 브라우저의 `al.Agent(llm, tools, system)` 에 해당합니다. 루프(LLM 판단 → 도구 → 관찰)는 LangGraph 가 만들어 줍니다.

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM = """당신은 친절한 날씨 비서입니다.
원칙: 도구 결과를 근거로만 답하고, 모르면 모른다고 말합니다. 답은 두 문장 이내로 짧게.
비가 오면 우산을, 기온이 낮으면 따뜻한 옷을 권합니다. 도시 이름은 영문으로 바꿔 도구에 넘깁니다."""

tools = [get_weather, recommend_outfit, calculator, search]
agent = create_react_agent(llm, tools, prompt=SYSTEM)

def show(result):
    """messages 를 브라우저의 trace() 처럼 출력"""
    for m in result["messages"]:
        if m.type == "ai" and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"🔧 {tc['name']}({tc['args']})")
        elif m.type == "tool":
            print(f"👁 {str(m.content)[:120]}")
        elif m.type == "ai":
            print(f"✅ {m.content}")

result = agent.invoke({"messages": [("user", "내일 부산 날씨 알려주고 우산이 필요한지 말해줘")]})
show(result)

In [ ]:
# 두 도구 연쇄: 날씨 → 옷차림 (실제 모델은 대본 없이 스스로 두 바퀴 돈다)
show(agent.invoke({"messages": [("user", "서울 날씨에 맞는 옷차림 알려줘")]}))

---
## 3단계. 멀티턴 대화 — `MemorySaver` + `thread_id`
📘 브라우저의 `agent.memory` 에 해당합니다. 같은 `thread_id` 로 부르면 이전 대화가 이어집니다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

agent_mem = create_react_agent(llm, tools, prompt=SYSTEM, checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "user-1"}}

for q in ["서울 날씨 어때?", "내 이름은 민수야", "그럼 도쿄는?", "내 이름이 뭐지?"]:
    r = agent_mem.invoke({"messages": [("user", q)]}, config)
    print("나 :", q)
    print("비서:", r["messages"][-1].content)

print("메모리 메시지 수:", len(r["messages"]))

---
## 4단계. 오류 처리 도구
📘 지원 도시 목록을 두고, 아닌 도시는 친절한 오류 메시지를 돌려줍니다. LLM 이 대안을 제시하는지 확인하세요.

In [ ]:
SUPPORTED = ["Seoul", "Busan", "Daegu", "Incheon", "Gwangju", "Daejeon", "Jeju"]

@tool
def city_weather(city: str) -> dict:
    """지원 도시(Seoul · Busan · Daegu · Incheon · Gwangju · Daejeon · Jeju)의 현재 날씨를 알려준다. city: 영문 도시 이름"""
    if city not in SUPPORTED:
        return {"error": f"{city} 는 지원하지 않는 도시입니다. 지원: " + ", ".join(SUPPORTED)}
    return get_weather.invoke({"city": city})

agent_safe = create_react_agent(llm, [city_weather, calculator], prompt=SYSTEM)
show(agent_safe.invoke({"messages": [("user", "도쿄 날씨 알려줘")]}))

---
## 5단계. 대화 로그 저장과 CLI 루프
📘 Colab 에서도 `input()` 이 동작합니다. “종료” 를 입력하면 대화 로그가 `chat_log.md` 로 저장됩니다.

In [ ]:
def run_cli(agent, thread_id="cli", max_turns=5):
    config = {"configurable": {"thread_id": thread_id}}
    log = ["# 대화 로그"]
    print('날씨 비서입니다. "종료" 를 입력하면 끝납니다.')
    for _ in range(max_turns):
        q = input("나: ").strip()
        if q in ("종료", "exit", ""):
            break
        a = agent.invoke({"messages": [("user", q)]}, config)["messages"][-1].content
        print("비서:", a)
        log += [f"- 나: {q}", f"- 비서: {a}"]
    with open("chat_log.md", "w", encoding="utf-8") as f:
        f.write("\n".join(log) + "\n")
    print("저장:", "chat_log.md", f"({len(log) // 2} 턴)")

run_cli(create_react_agent(llm, tools, prompt=SYSTEM, checkpointer=MemorySaver()))

---
# ✏️ 실습 문제

### 문제 1. 체감온도 도구
기온(°C)과 풍속(km/h)을 받아 `체감 = 기온 - 0.2 × 풍속` 을 소수점 한 자리로 돌려주는 `feels_like` 도구를 만들고 단독 테스트하세요.

In [ ]:
@tool
def feels_like(temperature: float, wind_kmh: float) -> dict:
    """기온과 풍속으로 체감온도를 계산한다. temperature: 섭씨 기온, wind_kmh: 풍속 km/h"""
    # ✏️ 여기에 코드를 작성하세요
    pass

print(feels_like.invoke({"temperature": 18.4, "wind_kmh": 2.1}))     # → 18.0
print(feels_like.invoke({"temperature": 5.0, "wind_kmh": 30.0}))     # → -1.0

### 문제 2. 등산 안내 페르소나
시스템 프롬프트를 “당신은 등산 안내 비서입니다 …” 로 바꾼 에이전트를 만들어 “강릉 날씨 알려줘” 에 답하게 하고, 호출된 도구 이름을 출력하세요.

In [ ]:
# ✏️ 여기에 코드를 작성하세요

### 문제 3. 환율 도구와 연쇄
고정 환율표 `{"USD": 1380.0, "JPY": 9.2, "EUR": 1490.0}` 로 `to_krw(amount, currency)` 도구를 만들고, 에이전트에 붙여 “100 달러는 원화로 얼마야?” 에 답하게 하세요. 모르는 통화는 `error` 를 돌려줍니다.

In [ ]:
RATES = {"USD": 1380.0, "JPY": 9.2, "EUR": 1490.0}

@tool
def to_krw(amount: float, currency: str) -> dict:
    """외화 금액을 원화(KRW)로 바꾼다. amount: 금액, currency: 통화 코드 (USD · JPY · EUR)"""
    # ✏️ 여기에 코드를 작성하세요
    pass

print(to_krw.invoke({"amount": 100, "currency": "USD"}))
# ✏️ 여기에 코드를 작성하세요

### 문제 4. 날씨 보고서 파일
서울 · 부산 · 제주 세 도시의 날씨를 에이전트에게 차례로 묻고, 결과를 마크다운 표(`| 도시 | 답 |`)로 `weather_report.md` 에 저장한 뒤 다시 읽어 출력하세요.

In [ ]:
# ✏️ 여기에 코드를 작성하세요

---
## 📝 정리
- 브라우저 `al.Agent` ↔ LangGraph `create_react_agent`, `al.ConversationMemory` ↔ `MemorySaver` + `thread_id`
- 도구는 `@tool` + docstring, 실패는 `{"error": …}` 로 돌려 루프가 멈추지 않게
- 실제 모델은 두 도구 연쇄와 “그럼 도쿄는?” 같은 맥락 질문을 스스로 처리한다
- 다음: 12차시 CrewAI 마케팅 팀